# Week 07 · Day 38 — NLP Foundations · Embeddings · Sentiment · Model Evaluation & Drift
**IIT Gandhinagar · Cohort 1**

**Topics:** Word2Vec, Polysemy, Disambiguation, Cosine Similarity, BOW, TF-IDF, Sentence-BERT

---

## 0. Setup & Imports

In [ ]:
# ─── Standard Library ───────────────────────────────────────────────────────
import os
import re
import warnings
import logging

# ─── Data & Numerics ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── NLP ────────────────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ─── Viz ────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.ERROR)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ─── Constants ──────────────────────────────────────────────────────────────
RANDOM_SEED     = 42
VECTOR_SIZE     = 100
MIN_COUNT       = 1
EPOCHS          = 20
WINDOW_SMALL    = 2
WINDOW_LARGE    = 10
SBERT_MODEL     = 'all-MiniLM-L6-v2'

np.random.seed(RANDOM_SEED)
print('✅ All imports successful')

## 1. Dataset Generation — ShopSense E-Commerce Reviews

In [ ]:
def generate_shopsense_reviews(n_reviews: int = 2000, random_state: int = RANDOM_SEED) -> pd.DataFrame:
    """
    Generate a synthetic ShopSense e-commerce review dataset with realistic
    polysemous usage of words like 'cheap'.

    Parameters
    ----------
    n_reviews   : int  — number of rows to generate
    random_state: int  — seed for reproducibility

    Returns
    -------
    pd.DataFrame with columns: review_id, review_text, rating, sentiment_label, category
    """
    np.random.seed(random_state)

    # Affordable context — cheap ≈ good value
    affordable_templates = [
        "This product is so cheap and affordable, great value for money!",
        "Incredibly cheap price for such good quality, highly recommended.",
        "Found it at a cheap price and it works perfectly well.",
        "Very cheap and budget-friendly option that does the job.",
        "Cheap deal compared to other brands, totally worth it.",
        "The price is quite cheap and the product is economical.",
        "Bought this as a cheap alternative and saved a lot of money.",
        "Such a cheap product that is also durable and efficient.",
        "Unbelievably cheap and works just as well as expensive brands.",
        "This affordable and cheap item exceeded my expectations.",
        "The budget-friendly price makes it cheap and accessible for all.",
        "Cheap in price but not in quality, very economical purchase.",
        "Got a cheap deal on this product, saved quite a bit of cash.",
        "This cheap item is great value for students on a tight budget.",
        "Cheap and cheerful, does exactly what it says on the tin.",
    ]

    # Low-quality context — cheap ≈ bad quality
    lowquality_templates = [
        "The product feels cheap and flimsy, not durable at all.",
        "Very cheap construction, broke within a week of use.",
        "Looks cheap and poorly made, definitely not worth it.",
        "Terrible product, feels cheap and falls apart easily.",
        "Cheap plastic material makes it fragile and unreliable.",
        "The cheap build quality is a big disappointment.",
        "Looks good in pictures but feels cheap and flimsy in person.",
        "Very cheap and poorly constructed, returned it immediately.",
        "Cheap workmanship, buttons fell off after two uses.",
        "This product is cheap in quality and broke after one month.",
        "Extremely cheap and low-quality material, total waste of money.",
        "Cheap looking design and the color faded after first wash.",
        "The product has a cheap feel to it, very disappointing.",
        "Cheap and nasty, do not waste your money on this product.",
        "Poorly made cheap item that does not match the description at all.",
    ]

    # General positive / negative reviews without 'cheap'
    positive_reviews = [
        "Absolutely love this product, works perfectly and looks amazing!",
        "Excellent quality and fast delivery, very satisfied with purchase.",
        "Great product for the price, highly recommend to everyone.",
        "Outstanding performance and beautiful design, five stars!",
        "Perfect fit and good quality material, would buy again.",
        "Superb product, arrived on time and in perfect condition.",
        "Incredible camera but the battery life could be better.",
        "Very satisfied with this purchase, exactly as described.",
        "Works great, easy to use and setup was straightforward.",
        "Amazing product, my whole family loves it, great buy!",
    ]

    negative_reviews = [
        "Battery drains fast, although the photos are stunning quality.",
        "Terrible quality, nothing like the pictures shown online.",
        "Very disappointed, stopped working after a few days.",
        "Poor customer service and low-quality product.",
        "Waste of money, do not recommend this to anyone.",
        "The product is flimsy and breaks easily, terrible build.",
        "Arrived damaged and the seller refused to help resolve.",
        "Does not work as advertised, very misleading product.",
        "Extremely disappointed with the quality of this item.",
        "Poor product, returned it within a day of receiving.",
    ]

    all_templates = affordable_templates + lowquality_templates + positive_reviews + negative_reviews
    categories    = ['Electronics', 'Clothing', 'Food', 'Home', 'Beauty', 'Books']

    reviews = []
    for i in range(n_reviews):
        text  = all_templates[i % len(all_templates)]
        # small lexical variation so Word2Vec sees more contexts
        if i % 7 == 0:
            text = text + " Overall happy with this purchase."
        if i % 11 == 0:
            text = "Update: " + text
        rating    = np.random.choice([1, 2, 3, 4, 5], p=[0.1, 0.1, 0.2, 0.3, 0.3])
        sentiment = 'positive' if rating >= 4 else ('neutral' if rating == 3 else 'negative')
        reviews.append({
            'review_id'      : f'R{i+1:05d}',
            'review_text'    : text,
            'rating'         : rating,
            'sentiment_label': sentiment,
            'category'       : np.random.choice(categories),
        })

    return pd.DataFrame(reviews)


df_reviews = generate_shopsense_reviews(n_reviews=2000)
print(f'Dataset shape : {df_reviews.shape}')
print(f'Sentiment dist:\n{df_reviews["sentiment_label"].value_counts()}')
df_reviews.head(3)

---
## Q1 — Word2Vec, Polysemy, and Window-Size Comparison

### Q1a. Train Word2Vec & Show ONE vector for 'cheap'

In [ ]:
def tokenize_corpus(texts: pd.Series) -> list:
    """
    Lowercase and tokenize a series of text strings.

    Parameters
    ----------
    texts : pd.Series of raw review strings

    Returns
    -------
    list of token lists (one per document)
    """
    tokenized = []
    for text in texts:
        tokens = word_tokenize(str(text).lower())
        tokens = [t for t in tokens if t.isalpha()]   # drop punctuation
        tokenized.append(tokens)
    return tokenized


def train_word2vec(
    sentences  : list,
    vector_size: int = VECTOR_SIZE,
    window     : int = 5,
    min_count  : int = MIN_COUNT,
    epochs     : int = EPOCHS,
    seed       : int = RANDOM_SEED,
) -> Word2Vec:
    """
    Train a skip-gram Word2Vec model on tokenized sentences.

    Parameters
    ----------
    sentences   : list of token lists
    vector_size : dimensionality of embeddings
    window      : context window size
    min_count   : minimum word frequency threshold
    epochs      : training iterations
    seed        : random seed for reproducibility

    Returns
    -------
    trained gensim Word2Vec model
    """
    model = Word2Vec(
        sentences   = sentences,
        vector_size = vector_size,
        window      = window,
        min_count   = min_count,
        workers     = 4,
        sg          = 1,       # skip-gram
        epochs      = epochs,
        seed        = seed,
    )
    return model


def compute_cosine(model: Word2Vec, word1: str, word2: str) -> float:
    """
    Compute cosine similarity between two words in a Word2Vec model.

    Parameters
    ----------
    model        : trained Word2Vec model
    word1, word2 : target words

    Returns
    -------
    float cosine similarity ∈ [-1, 1]
    """
    try:
        return float(model.wv.similarity(word1, word2))
    except KeyError as e:
        print(f'⚠️  Word not in vocabulary: {e}')
        return np.nan


# ── Train default model ──────────────────────────────────────────────────────
sentences    = tokenize_corpus(df_reviews['review_text'])
model_default = train_word2vec(sentences, window=5)

# Show ONE vector for 'cheap'
cheap_vector = model_default.wv['cheap']
print(f"'cheap' vector shape : {cheap_vector.shape}")
print(f"'cheap' vector (first 10 dims) : {cheap_vector[:10].round(4)}")
print()
print("Key insight: Word2Vec assigns EXACTLY ONE vector regardless of context.")
print("Whether 'cheap' means affordable or low-quality, the same embedding is used.")

# Cosine similarities
cos_affordable = compute_cosine(model_default, 'cheap', 'affordable')
cos_flimsy     = compute_cosine(model_default, 'cheap', 'flimsy')

print(f"\ncosine('cheap', 'affordable') = {cos_affordable:.4f}")
print(f"cosine('cheap', 'flimsy')     = {cos_flimsy:.4f}")
print()
if cos_affordable > cos_flimsy:
    print("→ 'affordable' meaning dominates in this corpus (affordable context is more frequent).")
else:
    print("→ 'low-quality' meaning dominates in this corpus.")

### Q1b. Disambiguation System — 'cheap' Meaning from Context

In [ ]:
def build_anchor_vectors(model: Word2Vec) -> dict:
    """
    Build anchor vectors for 'affordable' and 'low-quality' senses of 'cheap'
    by averaging embeddings of sense-specific anchor words.

    Parameters
    ----------
    model : trained Word2Vec model

    Returns
    -------
    dict with keys 'affordable' and 'low_quality', each a numpy array
    """
    affordable_anchors  = ['affordable', 'economical', 'budget', 'value', 'inexpensive', 'bargain', 'discount', 'deal', 'price', 'cost']
    low_quality_anchors = ['flimsy', 'poor', 'terrible', 'broke', 'fragile', 'disappointing', 'nasty', 'plastic', 'bad', 'fails']

    def mean_anchor(anchors):
        vecs = [model.wv[w] for w in anchors if w in model.wv]
        return np.mean(vecs, axis=0) if vecs else None

    return {
        'affordable' : mean_anchor(affordable_anchors),
        'low_quality': mean_anchor(low_quality_anchors),
    }


def get_context_vector(model: Word2Vec, sentence: str, target_word: str = 'cheap') -> np.ndarray | None:
    """
    Compute a context vector for `target_word` in a sentence by averaging
    the Word2Vec embeddings of all OTHER words in the sentence.

    Parameters
    ----------
    model        : trained Word2Vec model
    sentence     : input sentence string
    target_word  : the polysemous word to disambiguate

    Returns
    -------
    numpy array (context vector) or None if no context words found
    """
    tokens = word_tokenize(sentence.lower())
    tokens = [t for t in tokens if t.isalpha() and t != target_word]
    context_vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(context_vecs, axis=0) if context_vecs else None


def disambiguate_cheap(model: Word2Vec, sentence: str, anchors: dict) -> str:
    """
    Determine if 'cheap' in a sentence means 'affordable' or 'low-quality'
    by comparing the sentence context vector to anchor sense vectors.

    Parameters
    ----------
    model    : trained Word2Vec model
    sentence : input sentence containing 'cheap'
    anchors  : dict of sense → anchor vector (from build_anchor_vectors)

    Returns
    -------
    str — predicted sense: 'affordable' or 'low_quality'
    """
    if 'cheap' not in sentence.lower():
        return 'word not present'

    context_vec = get_context_vector(model, sentence)
    if context_vec is None:
        return 'insufficient context'

    scores = {}
    for sense, anchor_vec in anchors.items():
        if anchor_vec is not None:
            sim = cosine_similarity(
                context_vec.reshape(1, -1),
                anchor_vec.reshape(1, -1)
            )[0][0]
            scores[sense] = sim

    return max(scores, key=scores.get) if scores else 'unknown'


# ── Run disambiguation ───────────────────────────────────────────────────────
anchors = build_anchor_vectors(model_default)

test_sentences = [
    "This product is so cheap and affordable, great value for money!",
    "The product feels cheap and flimsy, not durable at all.",
    "Cheap deal compared to other brands, totally worth it.",
    "Very cheap construction, broke within a week of use.",
    "Found it at a cheap price and it works perfectly well.",
    "Looks cheap and poorly made, definitely not worth it.",
]

print("=" * 75)
print(f"{'Sentence':<55} {'Predicted Sense':<20}")
print("=" * 75)
for sent in test_sentences:
    prediction = disambiguate_cheap(model_default, sent, anchors)
    trunc = (sent[:52] + '...') if len(sent) > 55 else sent
    print(f"{trunc:<55} {prediction:<20}")
print("=" * 75)

print("""
Mechanism: The context vector (avg of non-target word embeddings) is compared
via cosine similarity to two anchor vectors:
  • 'affordable' anchor : avg(affordable, economical, budget, value, ...)
  • 'low_quality' anchor: avg(flimsy, poor, terrible, broke, ...)
The sense whose anchor is closest to the observed context wins.
""")

### Q1c. Window Size Comparison — Semantic vs Syntactic Relationships

In [ ]:
def compare_window_sizes(
    sentences    : list,
    probe_word   : str  = 'product',
    top_n        : int  = 10,
    window_small : int  = WINDOW_SMALL,
    window_large : int  = WINDOW_LARGE,
) -> pd.DataFrame:
    """
    Train two Word2Vec models with different window sizes and compare their
    nearest neighbours for a probe word.

    Parameters
    ----------
    sentences    : tokenised corpus
    probe_word   : word to inspect nearest neighbours for
    top_n        : how many nearest neighbours to retrieve
    window_small : small context window (captures syntactic)
    window_large : large context window (captures semantic)

    Returns
    -------
    pd.DataFrame with columns for each window's top-N neighbours
    """
    model_small = train_word2vec(sentences, window=window_small)
    model_large = train_word2vec(sentences, window=window_large)

    try:
        small_nn = [(w, round(s, 4)) for w, s in model_small.wv.most_similar(probe_word, topn=top_n)]
        large_nn = [(w, round(s, 4)) for w, s in model_large.wv.most_similar(probe_word, topn=top_n)]
    except KeyError:
        print(f"'{probe_word}' not in vocabulary")
        return pd.DataFrame()

    df_cmp = pd.DataFrame({
        f'window={window_small} (syntactic)' : [f"{w} ({s})" for w, s in small_nn],
        f'window={window_large} (semantic)'  : [f"{w} ({s})" for w, s in large_nn],
    })
    return df_cmp, model_small, model_large


result, model_small, model_large = compare_window_sizes(sentences, probe_word='cheap', top_n=8)

print("Nearest neighbours for 'cheap'")
print(result.to_string(index=False))

# Also compare cosine sims for both windows
pairs = [('cheap', 'affordable'), ('cheap', 'flimsy'), ('cheap', 'price'), ('cheap', 'material')]
print("\nCosine similarity comparison across window sizes:")
print(f"{'Pair':<30} {'Window=2':>10} {'Window=10':>10}")
print("-" * 52)
for w1, w2 in pairs:
    sim_small = compute_cosine(model_small, w1, w2)
    sim_large = compute_cosine(model_large, w1, w2)
    print(f"({w1}, {w2}){'':<{28 - len(w1) - len(w2)}} {sim_small:>10.4f} {sim_large:>10.4f}")

print("""
Analysis:
  window=2  (small) → captures SYNTACTIC patterns.
    Words that appear immediately before/after 'cheap' tend to be
    adjectives, determiners, or grammatical co-occurrences.

  window=10 (large) → captures SEMANTIC / topical patterns.
    Distant co-occurrences reveal broader meaning, so 'affordable',
    'value', 'budget' (or conversely 'flimsy', 'poor') emerge more clearly.
    However, noise also increases because many unrelated words share the window.
""")

In [ ]:
def visualize_window_comparison(model_small: Word2Vec, model_large: Word2Vec,
                                 target_words: list, title_prefix: str = 'PCA') -> None:
    """
    PCA 2-D projection of embeddings for a set of words,
    comparing small vs large window models side by side.

    Parameters
    ----------
    model_small, model_large : trained Word2Vec models
    target_words             : list of words to project
    title_prefix             : prefix for plot titles
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#e74c3c' if w in ['flimsy','poor','terrible','broke','fragile','nasty']
              else '#2ecc71' for w in target_words]

    for ax, (model, label) in zip(axes, [(model_small, f'window={WINDOW_SMALL}'),
                                          (model_large, f'window={WINDOW_LARGE}')]):
        valid  = [w for w in target_words if w in model.wv]
        vecs   = np.array([model.wv[w] for w in valid])
        coords = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(vecs)
        col    = [colors[target_words.index(w)] for w in valid]
        ax.scatter(coords[:, 0], coords[:, 1], c=col, s=80, alpha=0.8)
        for i, word in enumerate(valid):
            ax.annotate(word, (coords[i, 0], coords[i, 1]),
                        fontsize=9, ha='center', va='bottom')
        ax.set_title(f'{title_prefix} — {label}', fontsize=12)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        ax.grid(True, alpha=0.3)

    fig.suptitle("Green = affordable-sense anchors | Red = low-quality-sense anchors",
                 fontsize=10, y=0)
    plt.tight_layout()
    plt.savefig('window_pca_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Figure saved → window_pca_comparison.png")


probe_set = ['cheap', 'affordable', 'flimsy', 'value', 'poor', 'price',
             'budget', 'fragile', 'economical', 'plastic', 'deal', 'terrible']
visualize_window_comparison(model_small, model_large, probe_set)

---
## Q2 — Cosine Similarity: BOW vs TF-IDF vs Word2Vec vs Sentence-BERT

In [ ]:
# ─── Target reviews ─────────────────────────────────────────────────────────
REVIEW_A = "incredible camera but terrible battery life"
REVIEW_B = "Battery drains fast, although photos are stunning"

print("Review A:", REVIEW_A)
print("Review B:", REVIEW_B)
print("\nBoth express MIXED sentiment (good camera/photos, bad battery)")
print("Expectation: a good representation should show HIGH similarity.")

### Q2a. Compute Similarity with All Four Methods

In [ ]:
def bow_similarity(text_a: str, text_b: str) -> float:
    """
    Compute cosine similarity between two texts using Bag-of-Words (count) vectors.

    Parameters
    ----------
    text_a, text_b : raw text strings

    Returns
    -------
    float cosine similarity
    """
    vectorizer = CountVectorizer()
    try:
        vecs = vectorizer.fit_transform([text_a, text_b])
        return float(cosine_similarity(vecs[0], vecs[1])[0][0])
    except Exception as e:
        print(f'BOW error: {e}')
        return np.nan


def tfidf_similarity(text_a: str, text_b: str) -> float:
    """
    Compute cosine similarity between two texts using TF-IDF vectors.

    Parameters
    ----------
    text_a, text_b : raw text strings

    Returns
    -------
    float cosine similarity
    """
    vectorizer = TfidfVectorizer()
    try:
        vecs = vectorizer.fit_transform([text_a, text_b])
        return float(cosine_similarity(vecs[0], vecs[1])[0][0])
    except Exception as e:
        print(f'TF-IDF error: {e}')
        return np.nan


def word2vec_avg_similarity(model: Word2Vec, text_a: str, text_b: str) -> float:
    """
    Compute cosine similarity by averaging Word2Vec token vectors per document.

    Parameters
    ----------
    model          : trained Word2Vec model
    text_a, text_b : raw text strings

    Returns
    -------
    float cosine similarity
    """
    def doc_vector(text):
        tokens = [t for t in word_tokenize(text.lower()) if t.isalpha() and t in model.wv]
        return np.mean([model.wv[t] for t in tokens], axis=0) if tokens else None

    try:
        va, vb = doc_vector(text_a), doc_vector(text_b)
        if va is None or vb is None:
            return np.nan
        return float(cosine_similarity(va.reshape(1, -1), vb.reshape(1, -1))[0][0])
    except Exception as e:
        print(f'Word2Vec avg error: {e}')
        return np.nan


def sbert_similarity(text_a: str, text_b: str, model_name: str = SBERT_MODEL) -> float:
    """
    Compute cosine similarity using Sentence-BERT contextual embeddings.

    Parameters
    ----------
    text_a, text_b : raw text strings
    model_name     : HuggingFace model identifier for SentenceTransformer

    Returns
    -------
    float cosine similarity
    """
    try:
        sbert   = SentenceTransformer(model_name)
        embeddings = sbert.encode([text_a, text_b])
        return float(cosine_similarity(embeddings[0:1], embeddings[1:2])[0][0])
    except Exception as e:
        print(f'SBERT error: {e}')
        return np.nan


# ── Run all four ─────────────────────────────────────────────────────────────
sim_bow     = bow_similarity(REVIEW_A, REVIEW_B)
sim_tfidf   = tfidf_similarity(REVIEW_A, REVIEW_B)
sim_w2v     = word2vec_avg_similarity(model_default, REVIEW_A, REVIEW_B)
sim_sbert   = sbert_similarity(REVIEW_A, REVIEW_B)

results_df = pd.DataFrame({
    'Method'    : ['BOW', 'TF-IDF', 'Word2Vec (avg)', 'Sentence-BERT'],
    'Similarity': [sim_bow, sim_tfidf, sim_w2v, sim_sbert],
})
results_df['Correctly Identifies Similarity?'] = results_df['Similarity'].apply(
    lambda s: '✅ Yes' if s >= 0.5 else '❌ No'
)
print(results_df.to_string(index=False))

### Q2b. Word Overlap Analysis — BOW Failure

In [ ]:
def analyze_bow_overlap(text_a: str, text_b: str) -> None:
    """
    Walk through exact token overlap between two texts and explain
    why BOW fails to capture semantic similarity.

    Parameters
    ----------
    text_a, text_b : raw text strings
    """
    tokens_a = set(word_tokenize(text_a.lower()))
    tokens_b = set(word_tokenize(text_b.lower()))

    # Remove punctuation
    tokens_a = {t for t in tokens_a if t.isalpha()}
    tokens_b = {t for t in tokens_b if t.isalpha()}

    overlap    = tokens_a & tokens_b
    only_a     = tokens_a - tokens_b
    only_b     = tokens_b - tokens_a

    print("BOW Token Overlap Analysis")
    print("=" * 55)
    print(f"Review A tokens  : {sorted(tokens_a)}")
    print(f"Review B tokens  : {sorted(tokens_b)}")
    print(f"\nShared tokens    : {sorted(overlap) or 'NONE'}")
    print(f"Only in A        : {sorted(only_a)}")
    print(f"Only in B        : {sorted(only_b)}")
    print(f"\nJaccard overlap  : {len(overlap)}/{len(tokens_a | tokens_b)} = "
          f"{len(overlap)/len(tokens_a | tokens_b):.3f}")

    print("""
Explanation:
  Review A uses 'camera', 'terrible', 'battery', 'life'.
  Review B uses 'battery', 'drains', 'photos', 'stunning'.
  The only shared content word is 'battery'.
  BOW constructs orthogonal sparse vectors; cosine similarity is nearly 0.
  The semantic equivalences:
      camera ↔ photos     (same referent, different words)
      terrible ↔ drains   (both imply negative battery experience)
      incredible ↔ stunning (both mean very good)
  are INVISIBLE to BOW because it has no notion of lexical relationships.
""")

analyze_bow_overlap(REVIEW_A, REVIEW_B)

### Q2c. The Semantic Gap — How Each Method Closes It

In [ ]:
def explain_semantic_gap(results: pd.DataFrame) -> None:
    """
    Print a structured explanation of the semantic gap and how each
    representation method progressively narrows it.

    Parameters
    ----------
    results : DataFrame with 'Method' and 'Similarity' columns
    """
    explanations = {
        'BOW': (
            "Each word is an independent dimension. "
            "'camera' and 'photos' are orthogonal — cosine = 0. "
            "The semantic gap is FULLY OPEN."
        ),
        'TF-IDF': (
            "Same sparse token space as BOW, but rare words are up-weighted. "
            "Still no notion of synonymy. Gap barely narrows — "
            "unique words like 'stunning' remain invisible to each other."
        ),
        'Word2Vec (avg)': (
            "Words trained on context: 'camera' and 'photos' share similar "
            "distributional contexts so their vectors are close. "
            "Averaging captures partial semantics. Gap PARTIALLY CLOSED, "
            "but phrase-level / negation semantics are lost in averaging."
        ),
        'Sentence-BERT': (
            "A transformer encoder jointly encodes the full sentence. "
            "Contextual attention aligns 'camera/photos' and 'terrible/drains'. "
            "Fine-tuned on NLI + STS tasks. "
            "Gap is MOST CLOSED — both sentences end up in nearby "
            "regions of the embedding space."
        ),
    }

    print("\nSemantic Gap Analysis — Progressive Closure\n" + "=" * 60)
    for _, row in results.iterrows():
        method = row['Method']
        sim    = row['Similarity']
        desc   = explanations.get(method, '')
        bar    = '█' * int(sim * 20) + '░' * (20 - int(sim * 20))
        print(f"\n{method}  [score: {sim:.3f}]")
        print(f"  [{bar}]")
        print(f"  {desc}")

    print("""
Summary of progressive gap closure:
  BOW       → counts exact tokens   → gap fully open
  TF-IDF    → weights rare tokens   → gap negligibly reduced
  Word2Vec  → static distributed    → gap partially closed (synonym-level)
  SBERT     → contextual + fine-tuned → gap most closed (sentence-level)
""")

explain_semantic_gap(results_df)

### Visualisation — Similarity Comparison

In [ ]:
def plot_similarity_comparison(results: pd.DataFrame) -> None:
    """
    Bar chart comparing cosine similarity scores across all four methods.

    Parameters
    ----------
    results : DataFrame with 'Method' and 'Similarity' columns
    """
    palette = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(results['Method'], results['Similarity'],
                  color=palette, edgecolor='white', linewidth=0.5)
    ax.axhline(0.5, linestyle='--', color='grey', alpha=0.7, label='0.5 threshold')

    for bar, val in zip(bars, results['Similarity']):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01, f'{val:.3f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Cosine Similarity', fontsize=12)
    ax.set_title('Review A vs Review B — Similarity by Representation Method', fontsize=13)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('similarity_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Figure saved → similarity_comparison.png")

plot_similarity_comparison(results_df)

---
## Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║               WEEK 07 · DAY 38 — KEY TAKEAWAYS                            ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  Q1 — Word2Vec & Polysemy                                                 ║
║    • Word2Vec assigns ONE static vector per word type.                    ║
║    • 'cheap' is polysemous: both affordable (positive) and low-quality    ║
║      (negative) senses co-exist, but the model averages them away.        ║
║    • Context-based disambiguation uses sentence context vectors vs        ║
║      sense-anchored prototypes to recover the intended meaning.            ║
║    • window=2 : emphasises syntactic neighbours (adjacent grammar words)  ║
║      window=10: emphasises semantic topics (distant conceptual words)     ║
║                                                                            ║
║  Q2 — Semantic Gap                                                        ║
║    • BOW / TF-IDF are lexical-overlap methods → miss synonyms completely  ║
║    • Word2Vec averaging partially bridges the gap via static similarity   ║
║    • Sentence-BERT (contextual transformer) best closes the gap because   ║
║      it was fine-tuned to align semantically equivalent sentences         ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")